In [1]:
import polars as pl

raw = pl.scan_csv("../data/transactions.csv")
dup_ids = raw.group_by("txn_id").len().filter(pl.col("len") > 1)
dups = raw.join(dup_ids, on="txn_id", how="semi").collect(engine="streaming")
dups.sort("txn_id").head(20)

txn_id,txn_ts,branch_id,channel,txn_type,amount_uzs,client_segment
i64,str,str,str,str,i64,str
10000000387,"""2023-01-01 04:47:02""","""B050""","""ATM""","""withdrawal""",350000,"""payroll"""
10000000387,"""2023-01-01 04:47:02""","""B050""","""ATM""","""withdrawal""",350000,"""payroll"""
10000001587,"""2023-01-01 09:40:33""","""B050""","""ATM""","""withdrawal""",450000,"""retail"""
10000001587,"""2023-01-01 09:40:33""","""B050""","""ATM""","""withdrawal""",450000,"""retail"""
10000002477,"""2023-01-01 12:23:56""","""B037""","""ATM""","""withdrawal""",650000,"""payroll"""
…,…,…,…,…,…,…
10000004111,"""2023-01-01 16:51:51""","""B016""","""ATM""","""withdrawal""",450000,"""retail"""
10000004357,"""2023-01-01 17:37:03""","""B016""","""ATM""","""withdrawal""",700000,"""payroll"""
10000004357,"""2023-01-01 17:37:03""","""B016""","""ATM""","""withdrawal""",700000,"""payroll"""


In [4]:
extra = dups.filter(~pl.col("txn_id").is_first_distinct())
extra.height

125025

In [3]:
dups["txn_id"].n_unique()

125025

In [5]:
for col in ["channel", "txn_type", "client_segment"]:
    display(extra.group_by(col).len().sort("len", descending=True))

channel,len
str,u32
"""ATM""",77965
"""KASSA""",47060


txn_type,len
str,u32
"""withdrawal""",113177
"""deposit""",11848


client_segment,len
str,u32
"""retail""",97227
"""payroll""",15460
"""sme""",8593
"""premium""",3745


In [6]:
daily = pl.read_parquet("../data/daily.parquet")

def dup_share(key_extra, key_daily, name):
    """Доля лишних копий от всех операций, в процентах."""
    d = extra.group_by(key_extra.alias(name)).len().rename({"len": "n_dup"})
    t = daily.group_by(key_daily.alias(name)).agg(pl.col("n_txn").sum())
    return (
        t.join(d, on=name, how="left").fill_null(0)
        .with_columns((pl.col("n_dup") / pl.col("n_txn") * 100).round(3).alias("dup_pct"))
        .sort(name)
    )

by_branch = dup_share(pl.col("branch_id"), pl.col("branch_id"), "branch_id")
by_month = dup_share(pl.col("txn_ts").str.slice(0, 7), pl.col("date").dt.strftime("%Y-%m"), "month")

In [7]:
with pl.Config(tbl_rows=70):
    display(by_branch.sort("dup_pct", descending=True))
    display(by_month)

branch_id,n_txn,n_dup,dup_pct
str,u32,u32,f64
"""B033""",418866,1453,0.347
"""B042""",298678,983,0.329
"""B017""",772977,2513,0.325
"""B043""",308384,953,0.309
"""B034""",433725,1334,0.308
"""B024""",1932789,5936,0.307
"""B039""",592210,1816,0.307
"""B058""",507446,1559,0.307
"""B027""",438287,1337,0.305


month,n_txn,n_dup,dup_pct
str,u32,u32,f64
"""2023-01""",862209,1719,0.199
"""2023-02""",834688,1758,0.211
"""2023-03""",905309,1805,0.199
"""2023-04""",924067,1862,0.202
"""2023-05""",996572,1999,0.201
"""2023-06""",965972,1909,0.198
"""2023-07""",1010095,2008,0.199
"""2023-08""",1020440,2090,0.205
"""2023-09""",939095,1891,0.201


In [9]:
#Февраль 2025 по дням
feb = (
    extra.filter(pl.col("txn_ts").str.starts_with("2025-02"))
    .group_by(pl.col("txn_ts").str.slice(0, 10).alias("day"))
    .len()
    .sort("day")
)
with pl.Config(tbl_rows=31):
    display(feb)

day,len
str,u32
"""2025-02-01""",48
"""2025-02-02""",37
"""2025-02-03""",82
"""2025-02-04""",62
"""2025-02-05""",104
"""2025-02-06""",85
"""2025-02-07""",71
"""2025-02-08""",60
"""2025-02-09""",44


In [12]:
# Тестовые операции
raw.filter(pl.col("client_segment") == "internal_test").collect()

txn_id,txn_ts,branch_id,channel,txn_type,amount_uzs,client_segment
i64,str,str,str,str,i64,str
10013187635,"""2024-03-01 08:00:00""","""B033""","""ATM""","""withdrawal""",1000000000,"""internal_test"""
10013187665,"""2024-03-01 08:01:00""","""B033""","""ATM""","""withdrawal""",1000000000,"""internal_test"""
10013187696,"""2024-03-01 08:02:00""","""B033""","""ATM""","""withdrawal""",1000000000,"""internal_test"""
10013187722,"""2024-03-01 08:03:00""","""B033""","""ATM""","""withdrawal""",1000000000,"""internal_test"""
10013187744,"""2024-03-01 08:04:00""","""B033""","""ATM""","""withdrawal""",1000000000,"""internal_test"""
…,…,…,…,…,…,…
10038435972,"""2026-02-02 08:01:00""","""B060""","""ATM""","""withdrawal""",1000000000,"""internal_test"""
10038435989,"""2026-02-02 08:02:00""","""B060""","""ATM""","""withdrawal""",1000000000,"""internal_test"""
10038436013,"""2026-02-02 08:03:00""","""B060""","""ATM""","""withdrawal""",1000000000,"""internal_test"""
